In [1]:
import whisper
import torchaudio
import string
import numpy as np
import re
import matplotlib.pyplot as plt
import wave
from scipy.io import wavfile

from IPython.display import Audio, display
import pickle

import librosa
from scipy.signal import resample_poly
import jiwer
from whisper.normalizers import EnglishTextNormalizer

In [2]:
model = whisper.load_model("base.en")
print(
    f"Model is {'multilingual' if model.is_multilingual else 'English-only'} "
    f"and has {sum(np.prod(p.shape) for p in model.parameters()):,} parameters."
)

Model is English-only and has 71,825,408 parameters.


In [3]:
options = whisper.DecodingOptions(language = "en", without_timestamps = True)

In [4]:
audioFiles = np.load("/mnt/dataDrive/emg2Audio/cleanData/groundTruthAudioFiles.pkl", allow_pickle = True)

In [5]:
Audio(audioFiles[5], rate = 44100)

In [6]:
TARGET_SR = 16000  
ORIG_SR = 44100    

def toFloat32(x):
    x = np.asarray(x)
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(np.float32) / np.iinfo(x.dtype).max
    else:
        x = x.astype(np.float32)
    x -= x.mean()  
    peak = np.max(np.abs(x)) or 1.0
    if peak > 1.0:
        x /= peak
    return x

def resample(x):
    return resample_poly(x, up = 16000, down = 44100).astype(np.float32)

In [7]:
def transcribe(wave):
    x = toFloat32(wave)
    x16 = resample(x)
    out = model.transcribe(
        x16,
        language = "en",
        task = "transcribe",
        fp16 = False,  
        verbose = False
    )
    return {
        "text": out.get("text", "").strip(),
        "segments": [
            {"start": s["start"], "end": s["end"], "text": s["text"]}
            for s in out.get("segments", [])
        ],
        "language": out.get("language", "en")
    }

In [8]:
RESULTS = []
for i in range(len(audioFiles)):
    result = transcribe(audioFiles[i])["text"]
    clean = result.lower().translate(str.maketrans('', '', string.punctuation))
    RESULTS.append(clean)

100%|██████████| 257/257 [00:00<00:00, 4314.35frames/s]


In [10]:
with open("/mnt/dataDrive/emg2Audio/cleanData/textLABELS.pkl", "rb") as file:
    LABELS = pickle.load(file)

In [16]:
with open("groundTruthAudioTranscripts.pkl", "wb") as file:
    pickle.dump(RESULTS, file)

In [13]:
normalizer = EnglishTextNormalizer()

In [14]:
WER = []
for i in range(len(RESULTS)):
    ref = LABELS[i]
    hyp = RESULTS[i]

    werScore = jiwer.wer(normalizer(ref), normalizer(hyp))
    WER.append(werScore)

print(np.mean(WER))

0.14253641175567555


In [15]:
WER = []
for i in range(400):
    ref = LABELS[9260 + i]
    hyp = RESULTS[9260 + i]

    werScore = jiwer.wer(normalizer(ref), normalizer(hyp))
    WER.append(werScore)

print(np.mean(WER))

0.12661525974025975
